# DNB — Offline Smoke Tests (TWave)

TWave-style detection:

```
Wavelet phase at most recent sample → predict forward to target → stim
```

Phase map: `0=peak  π/2=falling  π=trough  3π/2=rising  2π=peak`

The detector reads phase at the trailing edge of the wavelet convolution,
computes time until the target phase, validates with multi-feature checks
(amplitude bounds, hi/lo frequency ratio, template matching), and emits
a candidate with the exact predicted timestamp.

**Event semantics:**
- `SLOW_WAVE` — detection. Carries `dt_to_stim_ms`, `phase_now`.
- `STIM` — scheduled stimulation at predicted target phase. `pulse_index` 1-indexed.

**Pipeline path tested here:**
Synthetic data is generated at the hardware rate (30 kHz) and processed
through the full pipeline including the downsampler (→ 500 Hz).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import (
    WaveletConvolution, TWaveDetector, AmplitudeMonitor, StimTrigger,
    Downsampler,
)
from dnb.validation.ground_truth import validate, Annotation
from test_data import TestData

import dnb
print(f'DNB v{dnb.__version__}')

td = TestData()

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
from math import pi

# ── TWave config — mirrors config.yaml ──
CFG = {
    'pipeline': {
        'sample_rate': 30000.0,
        'channel_id': 0,
        'buffer_duration': 10.0,
        'chunk_duration': 0.1,
    },
    'downsampler': {
        'enabled': True,
        'target_rate': 500.0,
    },
    'wavelet': {
        'freq_min': 0.5,
        'freq_max': 4.0,
        'n_freqs': 20,
        'n_cycles_base': 1.0,
    },
    'target_wave': {
        'id': 'slow_wave',
        'freq_range': (0.5, 2.0),
        'target_phase': 0.0,            # 0 = peak (up-state)
        'prediction_limit_s': 0.15,     # 150ms max lookahead
        'amp_min': 75.0,                # µV
        'amp_max': 300.0,               # µV
        'hilo_ratio_max': 0.15,         # IED rejection
        'hilo_boundary_hz': 10.0,
        'template_threshold': 0.8,      # sinusoidal match
        'template_window_s': 2.0,
        'warmup_chunks': 20,
    },
    'trigger': {
        'activation_detector_id': 'slow_wave',
        'inhibition_detector_id': 'ied_monitor',
        'n_pulses': 1,
        'backoff_s': 2.5,
        'inhibition_cooldown_s': 2.5,
    },
    'amplitude_monitor': {
        'enabled': True,
        'id': 'ied_monitor',
        'freq_range': (60.0, 120.0),
        'threshold': None,
        'adaptive_n_std': 5.0,
        'warmup_chunks': 20,
    },
    'synthetic': {
        'duration_s': 120.0,
        'snr': 5.0,
        'n_slow_waves': 15,
        'n_ieds': 10,
        'sw_amplitude': 500.0,
        'ied_amplitude': 3000.0,
        'seed': 42,
    },
    'validation': {
        'time_tolerance': 0.5,
    },
}

# Derived values
ANALYSIS_RATE = CFG['downsampler']['target_rate']
HARDWARE_RATE = CFG['pipeline']['sample_rate']
DS_FACTOR = int(round(HARDWARE_RATE / ANALYSIS_RATE))

print('Config loaded.')
print(f"  hardware_rate      = {HARDWARE_RATE:.0f} Hz")
print(f"  analysis_rate      = {ANALYSIS_RATE:.0f} Hz")
print(f"  target_phase       = {CFG['target_wave']['target_phase']:.2f} rad ({CFG['target_wave']['target_phase']*180/pi:.0f}°)")
print(f"  prediction_limit   = {CFG['target_wave']['prediction_limit_s']*1000:.0f} ms")
print(f"  amp_range          = [{CFG['target_wave']['amp_min']}, {CFG['target_wave']['amp_max']}] µV")
print(f"  hilo_ratio_max     = {CFG['target_wave']['hilo_ratio_max']}")
print(f"  template_threshold = {CFG['target_wave']['template_threshold']}")
print(f"  n_pulses           = {CFG['trigger']['n_pulses']}")
print(f"  chunk_duration     = {CFG['pipeline']['chunk_duration']}s")

In [ ]:
def make_pipeline_config():
    p = CFG['pipeline']
    return PipelineConfig(
        sample_rate=p['sample_rate'], channel_id=p.get('channel_id', 0),
        buffer_duration=p['buffer_duration'], chunk_duration=p['chunk_duration'],
    )

def make_downsampler():
    ds = CFG['downsampler']
    return Downsampler(target_rate=ds['target_rate'])

def make_wavelet():
    w = CFG['wavelet']
    return WaveletConvolution(
        freq_min=w['freq_min'], freq_max=w['freq_max'],
        n_freqs=w['n_freqs'], n_cycles_base=w['n_cycles_base'],
    )

def make_detector(**overrides):
    tw = {**CFG['target_wave'], **overrides}
    return TWaveDetector(
        id=tw['id'], freq_range=tw['freq_range'],
        target_phase=tw['target_phase'],
        prediction_limit_s=tw['prediction_limit_s'],
        amp_min=tw['amp_min'], amp_max=tw['amp_max'],
        hilo_ratio_max=tw.get('hilo_ratio_max'),
        hilo_boundary_hz=tw.get('hilo_boundary_hz', 10.0),
        template_threshold=tw.get('template_threshold'),
        template_window_s=tw.get('template_window_s', 2.0),
        warmup_chunks=tw['warmup_chunks'],
    )

def make_inhibitor(**overrides):
    am = {**CFG['amplitude_monitor'], **overrides}
    return AmplitudeMonitor(
        id=am['id'], freq_range=am['freq_range'],
        threshold=am['threshold'], adaptive_n_std=am['adaptive_n_std'],
        warmup_chunks=am['warmup_chunks'],
    )

def make_trigger(**overrides):
    tr = {**CFG['trigger'], **overrides}
    return StimTrigger(
        activation_detector_id=tr['activation_detector_id'],
        inhibition_detector_id=tr['inhibition_detector_id'],
        n_pulses=tr['n_pulses'],
        backoff_s=tr['backoff_s'],
        inhibition_cooldown_s=tr['inhibition_cooldown_s'],
    )

def make_standard_modules(**trigger_overrides):
    return [make_downsampler(), make_wavelet(), make_detector(), make_trigger(**trigger_overrides)]

def make_full_modules(**trigger_overrides):
    return [make_downsampler(), make_wavelet(), make_detector(), make_inhibitor(), make_trigger(**trigger_overrides)]

def get_detections(events):
    return [e for e in events if e.event_type == EventType.SLOW_WAVE]

def get_stims(events, pulse_index=None):
    out = [e for e in events if e.event_type == EventType.STIM]
    if pulse_index is not None:
        out = [e for e in out if e.metadata.get('pulse_index') == pulse_index]
    return out

def load_signal(path):
    data = np.load(str(path))
    sig = data['continuous']
    if sig.ndim == 2:
        sig = sig[0]
    return sig[::DS_FACTOR]

print('Helpers ready.')

---
## 1. Clean sine — TWave detection at peak

With a 1 Hz sine, the detector should read phase near 0 (peak) and
predict forward within the prediction limit. For a clean sine the
multi-feature validation needs relaxed thresholds (uniform amplitude
won't match absolute µV bounds from clinical data).

Data is generated at 30 kHz and processed through the downsampler.

In [ ]:
path = td.clean_sine(fs=HARDWARE_RATE)

# Clean sine: relax validation thresholds since it's synthetic
# - Wide amplitude range to accept the 500µV sine
# - Disable hilo ratio (no high-freq content)
# - Relax template threshold
# - Wider prediction limit to catch more detections
pipeline = Pipeline(
    source=FileSource(str(path)),
    modules=[
        make_downsampler(),
        make_wavelet(),
        make_detector(
            amp_min=0.0, amp_max=10000.0,
            hilo_ratio_max=None,
            template_threshold=None,
            prediction_limit_s=0.5,
        ),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=0.0),
    ],
    config=make_pipeline_config(),
)

events = pipeline.run_offline()
detections = get_detections(events)
stims = get_stims(events)

print(f'{len(detections)} detections, {len(stims)} stims')

if detections and stims:
    det_times = np.array([e.timestamp for e in detections])
    stim_times = np.array([e.timestamp for e in stims])
    det_phases = np.array([e.metadata['phase_now'] for e in detections])
    delays = np.array([e.metadata.get('dt_to_stim_ms', 0) for e in detections])

    target = CFG['target_wave']['target_phase']
    print(f'Phase at detection: mean={np.mean(det_phases):.3f} rad '
          f'(target={target:.3f})')
    print(f'Predicted delay to target: {np.mean(delays):.0f} \u00b1 {np.std(delays):.0f} ms')

    sig = load_signal(path)
    t = np.arange(len(sig)) / ANALYSIS_RATE

    stim_values = [sig[min(int(s * ANALYSIS_RATE), len(sig)-1)] for s in stim_times[:20]]
    print(f'Signal at stim times: mean={np.mean(stim_values):.0f} '
          f'(should be near +500, the peak)')

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(t, sig, 'b-', lw=0.5, alpha=0.7, label='signal')
    for i, e in enumerate(detections[:30]):
        axes[0].axvline(e.timestamp, color='blue', alpha=0.3, lw=1,
                        label='detection' if i == 0 else '')
    for i, e in enumerate(stims[:30]):
        axes[0].axvline(e.timestamp, color='red', alpha=0.6, lw=1,
                        label='stim' if i == 0 else '')
    axes[0].set_ylabel('Amplitude (\u00b5V)')
    axes[0].set_title(f'1 Hz sine \u2014 TWave detection, stim at predicted peak')
    axes[0].legend(loc='upper right', fontsize=8)

    axes[1].scatter(det_times[:30], det_phases[:30], c='blue', s=20, label='phase at detection')
    axes[1].axhline(target, color='green', ls='--',
                    label=f'target={target:.2f} rad ({target*180/pi:.0f}\u00b0)')
    axes[1].set_ylabel('Phase (rad)')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No detections or stims')

---
## 2. Synthetic slow waves — detection + stim validation

In [ ]:
syn = CFG['synthetic']
path_sw, gt_events = td.slow_waves(snr=syn['snr'], sample_rate=HARDWARE_RATE)
sw_gt = [e for e in gt_events if e.metadata.get('type') == 'SW']
print(f'Planted: {len(sw_gt)} slow waves')

sig = load_signal(path_sw)
t = np.arange(len(sig)) / ANALYSIS_RATE

# Relax thresholds for synthetic data (different amplitude scale)
pipeline = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_downsampler(),
        make_wavelet(),
        make_detector(
            amp_min=0.0, amp_max=10000.0,
            hilo_ratio_max=None,
            template_threshold=0.5,
            prediction_limit_s=0.3,
        ),
        make_trigger(inhibition_detector_id=None),
    ],
    config=make_pipeline_config(),
)
all_events = pipeline.run_offline()
detections = get_detections(all_events)
stims = get_stims(all_events)

print(f'Detections: {len(detections)}')
print(f'Stims (at predicted peak): {len(stims)}')

annotations = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt]
report = validate(stims, annotations, time_tolerance=CFG['validation']['time_tolerance'])
print(report.summary())

if detections:
    delays = [e.metadata.get('dt_to_stim_ms', 0) for e in detections]
    print(f'Detection\u2192Stim delay: {np.mean(delays):.0f} \u00b1 {np.std(delays):.0f} ms')

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5, label='signal')
for i, e in enumerate(sw_gt):
    ax.axvline(e.timestamp, color='green', alpha=0.4, lw=2,
               label='GT peak' if i == 0 else '')
for i, e in enumerate(detections):
    ax.axvline(e.timestamp, color='blue', alpha=0.3, lw=1,
               label='detection' if i == 0 else '')
for i, e in enumerate(stims):
    ax.axvline(e.timestamp, color='red', alpha=0.6, lw=1, ls='--',
               label='stim (predicted peak)' if i == 0 else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Green=GT peak, Blue=detection, Red=stim at predicted peak')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

---
## 3. N-pulse scheduling

- `n_pulses=0` → SLOW_WAVE only
- `n_pulses=1` → 1 stim at first predicted peak
- `n_pulses=3` → stim at first peak + 2 more at next predicted peaks

In [ ]:
for n_pulses in [0, 1, 3]:
    pipe = Pipeline(
        source=FileSource(str(path_sw)),
        modules=[
            make_downsampler(),
            make_wavelet(),
            make_detector(
                amp_min=0.0, amp_max=10000.0,
                hilo_ratio_max=None,
                template_threshold=0.5,
                prediction_limit_s=0.3,
            ),
            make_trigger(n_pulses=n_pulses, inhibition_detector_id=None),
        ],
        config=make_pipeline_config(),
    )
    dets = pipe.run_offline()
    sw_dets = get_detections(dets)
    all_stims = get_stims(dets)
    print(f'n_pulses={n_pulses}: {len(sw_dets)} detections, {len(all_stims)} stims')

    if n_pulses >= 1 and all_stims:
        for d in sw_dets[:2]:
            det_t = d.timestamp
            freq = d.metadata['frequency']
            delay_ms = d.metadata.get('dt_to_stim_ms', 0)
            my_stims = sorted(
                [s for s in all_stims if abs(s.metadata.get('detection_time', -1) - det_t) < 0.001],
                key=lambda e: e.metadata['pulse_index'],
            )
            print(f'  det t={det_t:.3f}s freq={freq:.2f}Hz delay_to_target={delay_ms:.0f}ms:')
            for s in my_stims:
                k = s.metadata['pulse_index']
                delay_from_det = (s.timestamp - det_t) * 1000
                print(f'    stim {k}/{n_pulses} t={s.timestamp:.3f}s '
                      f'(+{delay_from_det:.0f}ms from detection)')
    print()

In [ ]:
# 3-pulse plot
pipe_3 = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_downsampler(),
        make_wavelet(),
        make_detector(
            amp_min=0.0, amp_max=10000.0,
            hilo_ratio_max=None,
            template_threshold=0.5,
            prediction_limit_s=0.3,
        ),
        make_trigger(
            n_pulses=3, inhibition_detector_id=None,
            backoff_s=max(CFG['trigger']['backoff_s'], 5.0)),
    ],
    config=make_pipeline_config(),
)
dets_3 = pipe_3.run_offline()
det_events_3 = get_detections(dets_3)
stims_3 = get_stims(dets_3)

pulse_colors = {1: 'red', 2: 'orange', 3: 'purple'}
labels_used = set()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5, label='signal')
for i, e in enumerate(sw_gt):
    ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2,
               label='GT' if i == 0 else '')
for i, e in enumerate(det_events_3):
    ax.axvline(e.timestamp, color='blue', alpha=0.2, lw=1,
               label='detection' if i == 0 else '')
for e in stims_3:
    pi_idx = e.metadata['pulse_index']
    c = pulse_colors.get(pi_idx, 'gray')
    lbl = f'stim {pi_idx}' if pi_idx not in labels_used else ''
    labels_used.add(pi_idx)
    ax.axvline(e.timestamp, color=c, alpha=0.7, lw=1.2, ls='--', label=lbl)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('3-pulse: detect \u2192 stim at predicted peaks')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

### 3b. Verify stim timing

Stim k should land at `detection_time + dt_to_target + (k-1)/freq`.

In [ ]:
from collections import defaultdict

sequences = defaultdict(list)
for e in stims_3:
    sequences[e.metadata['detection_time']].append(e)

all_errors = []
for det_t, seq in sorted(sequences.items()):
    seq.sort(key=lambda e: e.metadata['pulse_index'])
    freq = seq[0].metadata['frequency']
    det_event = next((d for d in det_events_3 if abs(d.timestamp - det_t) < 0.001), None)
    if det_event is None:
        continue
    # First stim timestamp IS the predicted target time
    first_stim_t = seq[0].timestamp
    period = 1.0 / freq
    for e in seq:
        k = e.metadata['pulse_index']
        expected = first_stim_t + (k - 1) * period
        all_errors.append(e.timestamp - expected)

errs = np.array(all_errors) * 1000
print(f'{len(sequences)} sequences, {len(all_errors)} stim events')
print(f'Timing error: {np.mean(errs):.1f} \u00b1 {np.std(errs):.1f} ms')
print(f'  (stim timestamps are exact \u2014 no chunk quantisation)')

---
## 4. IED inhibition

TWave has two layers of IED rejection:
1. The hi/lo frequency ratio gate in the detector
2. The AmplitudeMonitor as a secondary safety net

Here we test with the AmplitudeMonitor active.

In [ ]:
path_ied, gt_ied = td.slow_waves_with_ieds(
    snr=syn['snr'], ied_amplitude=syn['ied_amplitude'],
    sample_rate=HARDWARE_RATE,
)
sw_gt_ied = [e for e in gt_ied if e.metadata.get('type') == 'SW']
ied_gt = [e for e in gt_ied if e.metadata.get('type') == 'IED']
print(f'Planted: {len(sw_gt_ied)} SWs, {len(ied_gt)} IEDs')

sig_ied = load_signal(path_ied)
t_ied = np.arange(len(sig_ied)) / ANALYSIS_RATE

# Relaxed detector for synthetic data + full inhibition
det_kwargs = dict(
    amp_min=0.0, amp_max=10000.0,
    hilo_ratio_max=None,
    template_threshold=0.5,
    prediction_limit_s=0.3,
)

# WITH inhibition
pipe_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        make_downsampler(), make_wavelet(),
        make_detector(**det_kwargs),
        make_inhibitor(),
        make_trigger(),
    ],
    config=make_pipeline_config(),
)
stims_inh = get_stims(pipe_inh.run_offline())

# WITHOUT inhibition
pipe_no = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        make_downsampler(), make_wavelet(),
        make_detector(**det_kwargs),
        make_trigger(inhibition_detector_id=None),
    ],
    config=make_pipeline_config(),
)
stims_no = get_stims(pipe_no.run_offline())

def near_ieds(stims, ieds, win=1.0):
    return sum(1 for s in stims if any(abs(s.timestamp - i.timestamp) < win for i in ieds))

print(f'With inhibition:    {len(stims_inh)} stims, {near_ieds(stims_inh, ied_gt)} near IEDs')
print(f'Without inhibition: {len(stims_no)} stims, {near_ieds(stims_no, ied_gt)} near IEDs')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, stims, label in [
    (axes[0], stims_no, 'WITHOUT inhibition'),
    (axes[1], stims_inh, 'WITH inhibition'),
]:
    ax.plot(t_ied, sig_ied, 'b-', lw=0.3, alpha=0.5, label='signal')
    for i, e in enumerate(sw_gt_ied):
        ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2,
                   label='GT SW' if i == 0 else '')
    for i, e in enumerate(ied_gt):
        ax.axvspan(e.timestamp - 0.1, e.timestamp + 0.1, color='orange', alpha=0.3,
                   label='IED' if i == 0 else '')
    for i, e in enumerate(stims):
        ax.axvline(e.timestamp, color='red', alpha=0.7, lw=1, ls='--',
                   label='stim' if i == 0 else '')
    ax.set_ylabel('Amp')
    ax.set_title(label)
    ax.legend(loc='upper right', fontsize=8)
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

---
## 5. Detection Report

Three-panel summary:
(a) stim-triggered average
(b) measured stim phase (independent Hilbert verification)
(c) inhibition summary

In [ ]:
det_kwargs_report = dict(
    amp_min=0.0, amp_max=10000.0,
    hilo_ratio_max=None,
    template_threshold=0.5,
    prediction_limit_s=0.3,
)

pipe_report = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        make_downsampler(), make_wavelet(),
        make_detector(**det_kwargs_report),
        make_inhibitor(),
        make_trigger(),
    ],
    config=make_pipeline_config(),
)
events_report = pipe_report.run_offline()
dets_report = get_detections(events_report)
stims_report = get_stims(events_report, pulse_index=1)

win_s = 2.0
win_samples = int(win_s * ANALYSIS_RATE)
t_win = np.arange(-win_samples, win_samples) / ANALYSIS_RATE

# Filter signal to SO band for stim-triggered analysis
from scipy.signal import butter, sosfilt

freq_range = CFG['target_wave']['freq_range']
nyq = ANALYSIS_RATE / 2.0
sos = butter(4, [freq_range[0] / nyq, freq_range[1] / nyq], btype='band', output='sos')
sig_ied_so = sosfilt(sos, sig_ied)

# (a) Extract stim-triggered windows
stim_windows = []
for e in stims_report:
    centre = int(e.timestamp * ANALYSIS_RATE)
    lo, hi = centre - win_samples, centre + win_samples
    if lo >= 0 and hi < len(sig_ied):
        stim_windows.append(sig_ied_so[lo:hi])

# (b) Per-trial phase from filtered windows
stim_phases = []
if stim_windows:
    centre = len(stim_windows[0]) // 2
    half_cycle = int(ANALYSIS_RATE / 2)
    lo_w = max(0, centre - half_cycle)
    hi_w = min(len(stim_windows[0]), centre + half_cycle)
    for w in stim_windows:
        peak_idx = lo_w + np.argmax(w[lo_w:hi_w])
        dt = (peak_idx - centre) / ANALYSIS_RATE
        stim_phases.append((-dt * 2 * pi) % (2 * pi))
    stim_phases = np.array(stim_phases)

# (c) Inhibition comparison
cooldown = CFG['trigger']['inhibition_cooldown_s']
pipe_noinh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        make_downsampler(), make_wavelet(),
        make_detector(**det_kwargs_report),
        make_trigger(inhibition_detector_id=None),
    ],
    config=make_pipeline_config(),
)
stims_noinh = get_stims(pipe_noinh.run_offline(), pulse_index=1)

blocked_stims = []
for s in stims_noinh:
    if not any(abs(s2.timestamp - s.timestamp) < 0.1 for s2 in stims_report):
        nearest_ied = min((abs(s.timestamp - ied.timestamp) for ied in ied_gt), default=999)
        blocked_stims.append((s.timestamp, nearest_ied))

correct_blocks = sum(1 for _, d in blocked_stims if d < cooldown)
wrong_blocks = sum(1 for _, d in blocked_stims if d >= cooldown)

print(f'Without inhibition: {len(stims_noinh)} stims')
print(f'With inhibition:    {len(stims_report)} stims')
print(f'Blocked: {len(blocked_stims)} ({correct_blocks} near IED, {wrong_blocks} no nearby IED)')

# ── Plot ──────────────────────────────────────────────────────────────
target_phase = CFG['target_wave']['target_phase']
fig = plt.figure(figsize=(18, 5))

# (a) Stim-triggered average
ax_a = fig.add_subplot(1, 3, 1)
for w in stim_windows:
    ax_a.plot(t_win, w, alpha=0.3, lw=0.5)
if stim_windows:
    avg = np.mean(stim_windows, axis=0)
    ax_a.plot(t_win, avg, 'k-', lw=2, label='mean')
ax_a.axvline(0, color='red', ls='--', alpha=0.6, lw=1, label='stim')
ax_a.set_xlabel('Time from stim (s)')
ax_a.set_ylabel('Amplitude')
ax_a.set_title(f'Stim-triggered average (n={len(stim_windows)})')
ax_a.legend(fontsize=8)

# (b) Stim phase
ax_b = fig.add_subplot(1, 3, 2, projection='polar')
if len(stim_phases) > 0:
    n_bins = 24
    counts, edges = np.histogram(stim_phases, bins=n_bins, range=(0, 2*pi))
    centres = (edges[:-1] + edges[1:]) / 2
    width = 2 * pi / n_bins
    ax_b.bar(centres, counts, width=width, alpha=0.6, edgecolor='k', lw=0.5,
             color='#E74C3C')

    mean_phase = np.angle(np.mean(np.exp(1j * stim_phases))) % (2 * pi)

    ax_b.axvline(target_phase, color='green', ls='--', lw=2)
    ax_b.axvline(mean_phase, color='darkred', ls='-', lw=2)

    ax_b.set_thetagrids([0, 90, 180, 270],
                         labels=['peak', 'falling', 'trough', 'rising'], fontsize=9)
    ax_b.set_yticklabels([])

    error_deg = abs(mean_phase - target_phase) * 180 / pi
    if error_deg > 180:
        error_deg = 360 - error_deg

    ax_b.set_title(f'Stim phase (n={len(stim_phases)})\n'
                   f'green=target  red=actual  error={error_deg:.0f}\u00b0',
                   fontsize=9, pad=15)
else:
    ax_b.set_title('No stims')

# (c) Inhibition summary
ax_c = fig.add_subplot(1, 3, 3)
labels = ['Fired\n(no IED)', 'Blocked\n(IED nearby)', 'Blocked\n(no IED)']
values = [len(stims_report), correct_blocks, wrong_blocks]
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax_c.bar(labels, values, color=colors, edgecolor='k', lw=0.5)
for bar, val in zip(bars, values):
    if val > 0:
        ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                  str(val), ha='center', va='bottom', fontweight='bold')
ax_c.set_ylabel('Count')
ax_c.set_title(f'Inhibition (cooldown={cooldown}s)')
ax_c.set_ylim(0, max(values + [1]) * 1.3)

plt.tight_layout()
plt.show()